In [1]:
import numpy as np
from time import time
from HARK.models import HabitPortfolioConsumerType, RiskyAssetConsumerType
from HARK.ConsumptionSaving.ConsHabitModel import (
    HabitPortfolioConsumerType_defaults,
)

mystr = lambda x: "{:.3f}".format(x)

In [2]:
# Make a parameter dictionary
my_params = HabitPortfolioConsumerType_defaults.copy()
del my_params["constructors"]  # don't want to overwrite these
adjusted_params = {
    "CRRA": 3.5,
    "LivPrb": [1.0],
    "DiscFac": 0.94,
    "PermGroFac": [1.00],
    "Rfree": [1.01],
    "RiskyAvg": 1.04,
    "RiskyStd": 0.18,
    "RiskyShareFixed": None,
    "hLogInitMean": 0.0,
    "HabitMax": 8.0,
    "HabitCount": 51,
    "cycles": 0,
}
my_params.update(adjusted_params)

In [3]:
# Define grid specifications
wealth_grid = {"min": 0.0, "max": 120.0, "N": 201, "order": 2.5}
con_grid = {"min": 0.0, "max": 3.0, "N": 301, "order": 1.1}
share_grid = {"min": 0.0, "max": 1.0, "N": 101}
habit_grid = {"min": 0.3, "max": 3.0, "N": 76, "order": 2.0}
my_grids_base = {
    "kNrm": wealth_grid,
    "wNrm": wealth_grid,
    "qNrm": wealth_grid,
    "cNrm": con_grid,
    "Share": share_grid,
}
my_grids = my_grids_base.copy()
my_grids["hPre"] = habit_grid
my_grids["hNrm"] = habit_grid

In [4]:
# Make and solve the base type with no habits
BaseType = RiskyAssetConsumerType(**my_params)
t0 = time()
BaseType.solve()
BaseType.initialize_sym()
BaseType._simulator.make_transition_matrices(my_grids_base, norm="PermShk")
BaseType._simulator.find_steady_state()
t1 = time()
print("Solving the model with no habits took " + mystr(t1 - t0) + " seconds.")

Solving the model with no habits took 3.392 seconds.


In [5]:
# Calculate target assets and risky share
a_targ = BaseType._simulator.get_long_run_average("aNrm")
w_targ = BaseType._simulator.get_long_run_average("wNrm")
q_targ = BaseType._simulator.get_long_run_average("qNrm")
s_targ = q_targ / w_targ
print(a_targ, s_targ)

12.552368247843692 0.6945006925357814


In [6]:
# Define a function that returns the weighted distance of long run averages
def calc_distance(alpha, beta, lamda, rho):
    temp_params = my_params.copy()
    temp_params["HabitWgt"] = alpha
    temp_params["DiscFac"] = beta
    temp_params["HabitRte"] = lamda
    temp_params["CRRA"] = rho
    TempType = HabitPortfolioConsumerType(**temp_params)

    TempType.solve()
    TempType.initialize_sym()
    TempType._simulator.make_transition_matrices(my_grids, norm="PermShk")
    TempType._simulator.find_steady_state()
    a_val = TempType._simulator.get_long_run_average("aNrm")
    w_val = TempType._simulator.get_long_run_average("wNrm")
    q_val = TempType._simulator.get_long_run_average("qNrm")
    s_val = q_val / w_val

    distance = np.sqrt((a_val - a_targ) ** 2 + (10 * (s_val - s_targ)) ** 2)
    print(alpha, beta, lamda, rho, a_val, s_val)
    return distance

In [10]:
# Define a temporary function to minimize
def my_func(x):
    varphi = x[0]
    rho = x[1]
    beta = 1.0 / (1.0 + np.exp(varphi))
    return calc_distance(0.8, beta, 0.1, rho)

In [11]:
# Minimize the distance
# out = minimize_nelder_mead(my_func, [-2.7, 5.9])

0.8 0.9370266439430035 0.1 5.9 16.656265235948087 0.566139355037996
0.8 0.9445381143324907 0.1 5.9 20.374935738845693 0.519457948977061
0.8 0.9370266439430035 0.1 6.195 17.531153490978813 0.5384303129913331
0.8 0.9285747874424048 0.1 6.195 14.029915965478295 0.5883926489398816
0.8 0.9190865327845347 0.1 6.342500000000001 11.247858352057884 0.6268385906237995
0.8 0.9190865327845347 0.1 6.047500000000001 10.406199623511663 0.6591162137637413
0.8 0.8965995485417505 0.1 6.490000000000002 6.581300487378978 0.7182989171903681
0.8 0.9285747874424048 0.1 6.047500000000001 13.585657568124892 0.6034373114213455
0.8 0.9285747874424048 0.1 6.342500000000001 14.46612341797572 0.5737038924521108
0.8 0.9215611649074514 0.1 6.121250000000002 11.373303365205947 0.6368714227307243
0.8 0.9307810884424494 0.1 5.826250000000002 13.750492731217914 0.6129271049197286
0.8 0.9280131540418358 0.1 5.955312500000002 13.09009506720205 0.6164793522582592
0.8 0.9209490767090109 0.1 6.029062500000002 10.9109746135577


KeyboardInterrupt



In [9]:
1.0 / (1.0 + np.exp(-2.55))

0.9275735146384823